# Quickstart: `ctkit`This notebook walks through the package end to end: pick a dataset, download it,throw away the series you cannot use, process the rest with a named protocol, andextract features.The guiding idea is that a **protocol is an object**. You can print it, save it toYAML, publish it with a paper, and hand it to someone else to reproduce your datasetexactly.Install with:```pip install 'ctkit[all]'```

In [ ]:
try:
    import ctkit
except ImportError:
    %pip install -q "ctkit[all]"

import ctkit
print(ctkit.__version__)

## 1. What is availableThe catalogue holds the TCGA and CPTAC collections with curated settings — which organsto segment, which intensity window to clip to, what output size to standardize to — pluswidely used public CT collections.Any TCIA collection works even if it is not in the catalogue; `list_collections()` asksthe archive directly.

In [ ]:
from ctkit import list_datasets

list_datasets().head(15)

In [ ]:
from ctkit import list_collections

collections = list_collections()
print(len(collections), "collections on TCIA")
[name for name in collections if "TCGA" in name][:8]

## 2. Choose a protocol`ProcessingConfig` is the whole protocol in one object. `for_dataset` fills in thesettings curated for a collection; every field can be overridden.`describe()` prints it as prose — this is the paragraph that belongs in a methodssection.

In [ ]:
from ctkit import ProcessingConfig

config = ProcessingConfig.for_dataset("tcga-kich")
print(config.describe())

In [ ]:
# Every knob is a plain field, and any of them can be changed.
config = config.replace(target_spacing=(1.0, 1.0, 3.0), fast_segmentation=True)

print("steps:            ", config.steps)
print("clip window:      ", (config.clip_min, config.clip_max))
print("organs:           ", config.organs)
print("needs cohort pass:", config.needs_dataset_pass)

## 3. Download`download()` uses the TCIA REST API, so no Java client, manifest file or login isneeded for public collections.Series metadata is fetched first, and anything that fails the header checks — localizers,scouts, topograms, thick slices — is skipped **before** its pixels are downloaded. Eachseries is converted to NIfTI in a temporary directory; only the converted volume is kept.`limit=3` keeps this notebook quick; drop it for the full collection.

In [ ]:
from ctkit import download

raw = download("tcga-kich", "data/raw", limit=3, modality="CT", workers=3)
raw

In [ ]:
import pandas as pd

# Why series were dropped before download.
excluded = pd.read_csv("data/raw/excluded_series.csv")
excluded["exclusion_reason"].value_counts()

## 4. Look at a scan`RadiologyImage` is one scan plus an optional mask, held in memory. It accepts a NIfTIpath, a DICOM directory or zip, a `.npy` file, or an already-loaded array.

In [ ]:
image = raw[0]
print(image)
image.statistics()

In [ ]:
image.plot(window=(-200, 300));

## 5. Process one imageEach step is a method, and they chain. Nothing touches the disk until `.save()`.Note what each step changes: `orient` fixes the axis order, `resample` changes the shapebecause the voxel size changed, `apply_mask` crops to the region of interest.

In [ ]:
from ctkit import RadiologyImage

step = RadiologyImage(image.source, series_id=image.series_id)
print("loaded    ", step.shape, step.orientation, step.spacing)

step.orient()
print("oriented  ", step.shape, step.orientation)

step.clip(-200, 300)
print("clipped   ", (float(step.array.min()), float(step.array.max())))

step.resample((1.0, 1.0, 3.0))
print("resampled ", step.shape, step.spacing)

### SegmentationThe collection ships no tumor masks, so organ masks come from TotalSegmentator. It runsin a temporary directory that is deleted as soon as the masks are in memory.Organs are labelled 1 and tumor 2. If a tumor mask is present, only the organs thatactually contain tumor are kept, so the healthy contralateral kidney does not enter theregion of interest.This is the slow step — the first run also downloads model weights.

In [ ]:
step.segment(organs=config.organs, fast=True)
print("mask labels:", step.labels, "| voxels:", int((step.mask_array > 0).sum()))
step.plot(window=(-200, 300));

In [ ]:
step.apply_mask(crop=True, padding=5)
print("masked and cropped:", step.shape)
step.plot();

In [ ]:
# The full chain that produced this image, with the parameters used.
step.history

## 6. Process a cohort`Dataset` holds paths rather than pixels and loads one image at a time, so a cohort thatwould not fit in memory still works.First, quality control. The report covers every series, including the ones that passed,with the measurements behind each decision — this is what you cite when a paper asks howmany scans were excluded and why.

In [ ]:
from ctkit import Dataset, QCCriteria

dataset = Dataset.from_directory("data/raw")
usable = dataset.filter(QCCriteria(min_slices=25))
usable.qc_report

Then process. Only the final image and mask are written, alongside the config that
produced them and a manifest.

`target_shape=None` means "use the 95th percentile shape of this cohort", which the
dataset measures for itself. That measurement needs a look at the processed data before
the final write, so the pipeline splits itself: the per-image steps (including
segmentation) run once into a temporary directory, the cohort statistics are measured
from that, and only the cheap crop/pad and z-scoring steps are applied to produce the
final files. The temporary directory is deleted before `process()` returns.

Pin `target_shape` and `dataset_mean`/`dataset_std` if you would rather run in a single
pass with no staging at all.

In [ ]:
processed = usable.process(
    config.replace(target_shape=None),
    out_dir="data/processed",
    workers=1,          # keep at 1 when segmenting on one GPU
)
processed

In [ ]:
import os

for entry in sorted(os.listdir("data/processed")):
    print(entry)

In [ ]:
# Every case ends up the same shape, ready to stack into a tensor.
[img.shape for img in processed]

## 7. Radiomic featuresPyRadiomics does its own resampling and normalization, so the `radiomics` preset turnsoff the steps that would otherwise be applied twice.`labels=[1, 2]` measures organ and tumor together; `labels=[2]` would be tumor only.

In [ ]:
features = processed.radiomics(labels=[1, 2], out_csv="data/features.csv")
print(features.shape)
features.filter(regex="series_id|original_firstorder").head()

## 8. Reproducing this laterThe config written next to the output is enough to rebuild the dataset from the rawimages — no notebook state required.

In [ ]:
print(open("data/processed/processing_config.yaml").read()[:600])

In [ ]:
saved = ProcessingConfig.from_yaml("data/processed/processing_config.yaml")
print(saved.describe())

## From the command lineThe same operations without writing any Python:```shctkit datasetsctkit download tcga-kich --out data/raw --limit 20ctkit filter data/raw --report qc.csvctkit process data/raw --out data/processed --dataset tcga-kichctkit radiomics data/processed --out features.csvctkit info data/processed/TCGA-KM-8438/imaging.nii.gz```## Where to go next- `ProcessingConfig` — every step and its parameters- `QCCriteria` — the exclusion thresholds- `Dataset.from_metadata` — drive a cohort from a metadata table- `dimensionality="2D"` — keep the slice with the most tumor instead of the volume- The original protocol notebook, `tcia_ct_processing_protocol.ipynb`, shows the same  pipeline written out step by step.